# Probability Distributions

Wiki reference for [probability distributions](https://ml-viz-ruby.vercel.app/wiki/probability-distributions).

**The idea in one sentence.** The common distributions form a connected family — the Binomial
**becomes** Poisson as $n\to\infty$ with $np$ fixed, sample means of *anything* become Gaussian by
the **central limit theorem** — but the tails differ enormously, so a "rare" event under a
Gaussian can be common under a heavy-tailed distribution.

We work through the discrete and continuous families from scratch, **validate the
Binomial→Poisson limit and the CLT**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
plt.style.use('dark_background')
rng = np.random.default_rng(0)

## 1 — Discrete families (PMFs)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
k = np.arange(0, 16)
ax[0].bar(k, stats.binom.pmf(k, n=15, p=0.4), color='#6366f1')
ax[0].set_title('Binomial(n=15, p=0.4)\ncounts of successes')
ax[1].bar(k, stats.poisson.pmf(k, mu=4), color='#34d399')
ax[1].set_title('Poisson(λ=4)\nrare-event counts (mean=var=λ)')
ax[2].bar(k, stats.geom.pmf(k, p=0.3), color='#f59e0b')
ax[2].set_title('Geometric(p=0.3)\ntrials until first success')
for a in ax: a.set_xlabel('k'); a.set_ylabel('P(X=k)')
plt.tight_layout(); plt.show()

### Validate: pmfs are valid and Poisson has mean = variance

A probability mass function sums to 1, and the Poisson distribution has the special property
mean = variance = $\lambda$. We confirm both.

In [ ]:
total = stats.binom.pmf(np.arange(0, 16), n=15, p=0.4).sum()
mean_p, var_p = stats.poisson.stats(mu=4, moments='mv')
print(f'Binomial pmf sums to {total:.4f};  Poisson(4) mean={float(mean_p):.1f}, var={float(var_p):.1f}')
assert np.isclose(total, 1.0, atol=1e-6), 'a probability mass function sums to 1'
assert np.isclose(float(mean_p), float(var_p)), 'the Poisson distribution has mean = variance = lambda'
print('\n✅ valid pmfs; Poisson is the mean-equals-variance count distribution')

## 2 — Continuous families (PDFs)

Note how Beta lives on [0,1] (a distribution over a probability) and Student-t has heavier tails than Gaussian.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
x = np.linspace(-5, 5, 400)
ax[0].plot(x, stats.norm.pdf(x), label='Gaussian', color='#6366f1')
ax[0].plot(x, stats.laplace.pdf(x), label='Laplace (heavier)', color='#f59e0b')
ax[0].plot(x, stats.t.pdf(x, df=2), label='Student-t df=2 (heaviest)', color='#f87171')
ax[0].set_title('Symmetric: tails matter'); ax[0].legend(fontsize=8)

xp = np.linspace(0, 5, 400)
ax[1].plot(xp, stats.expon.pdf(xp, scale=1), label='Exponential', color='#34d399')
ax[1].plot(xp, stats.gamma.pdf(xp, a=2), label='Gamma(2)', color='#6366f1')
ax[1].set_title('Positive-only: durations'); ax[1].legend(fontsize=8)

xb = np.linspace(0, 1, 400)
for (a_, b_), c in [((2,2),'#6366f1'), ((5,2),'#34d399'), ((1,3),'#f59e0b')]:
    ax[2].plot(xb, stats.beta.pdf(xb, a_, b_), label=f'Beta({a_},{b_})', color=c)
ax[2].set_title('Beta: a distribution over a probability'); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3 — Relationship: Binomial → Poisson

For large n, small p with np=λ fixed, the Binomial converges to the Poisson (the rare-event limit).

In [ ]:
lam = 3.0
k = np.arange(0, 12)
plt.figure(figsize=(8, 4))
plt.bar(k, stats.poisson.pmf(k, lam), alpha=0.5, color='#34d399', label='Poisson(λ=3)')
for n in [6, 30, 300]:
    plt.plot(k, stats.binom.pmf(k, n=n, p=lam/n), 'o-', ms=4, label=f'Binomial(n={n}, p={lam/n:.3f})')
plt.xlabel('k'); plt.ylabel('P(X=k)'); plt.legend()
plt.title('Binomial → Poisson as n grows (np=λ fixed)'); plt.tight_layout(); plt.show()

### Validate: Binomial converges to Poisson

For rare events, $\text{Binomial}(n, \lambda/n) \to \text{Poisson}(\lambda)$ as $n$ grows. We
confirm the approximation error shrinks as $n$ increases with $np = \lambda$ held fixed.

In [ ]:
k = np.arange(0, 12)
err_small = np.abs(stats.binom.pmf(k, 6, 3/6) - stats.poisson.pmf(k, 3)).max()
err_large = np.abs(stats.binom.pmf(k, 300, 3/300) - stats.poisson.pmf(k, 3)).max()
print(f'max |Binomial - Poisson|: n=6 -> {err_small:.3f},  n=300 -> {err_large:.4f}')
assert err_large < err_small, 'Binomial(n, lambda/n) converges to Poisson(lambda) as n grows'
print('\n✅ the Poisson is the large-n, rare-event limit of the Binomial')

## 4 — Relationship: Central Limit Theorem → Gaussian

Averages of samples from *any* distribution become Gaussian. Here we average draws from a skewed Exponential.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
for a, m in zip(ax, [1, 5, 30]):
    means = rng.exponential(1.0, size=(20000, m)).mean(axis=1)
    a.hist(means, bins=60, density=True, color='#6366f1', alpha=0.8)
    # overlay the Gaussian the CLT predicts (Exp(1): mean=1, var=1 -> var of mean = 1/m)
    g = np.linspace(means.min(), means.max(), 200)
    a.plot(g, stats.norm.pdf(g, 1.0, np.sqrt(1.0/m)), color='#f59e0b', lw=2)
    a.set_title(f'mean of m={m} Exp(1) draws')
plt.suptitle('CLT: sample means become Gaussian regardless of the source shape', y=1.03)
plt.tight_layout(); plt.show()

### Validate: the central limit theorem

The sample mean of i.i.d. draws (here from a skewed Exponential) becomes Gaussian as the sample
size $m$ grows, with standard deviation shrinking as $\sigma/\sqrt m$. We confirm the tightening
matches the CLT prediction.

In [ ]:
m1 = rng.exponential(1.0, (20000, 1)).mean(axis=1)
m30 = rng.exponential(1.0, (20000, 30)).mean(axis=1)
print(f'std of the sample mean: m=1 -> {m1.std():.3f},  m=30 -> {m30.std():.3f}  (CLT predicts {1/np.sqrt(30):.3f})')
assert m30.std() < m1.std(), 'the sampling distribution of the mean tightens as m grows'
assert abs(m30.std() - 1/np.sqrt(30)) < 0.02, 'and matches the CLT prediction sigma/sqrt(m)'
print('\n✅ averages of anything become Gaussian (CLT) with std shrinking as 1/sqrt(m)')

## 5 — Conjugacy: Beta prior + Bernoulli data → Beta posterior

The closed-form Bayesian update behind A/B testing: no MCMC needed.

In [ ]:
# True conversion rate 0.30; observe data and watch the Beta posterior sharpen
true_p = 0.30
alpha0, beta0 = 1, 1   # uniform prior = Beta(1,1)
x = np.linspace(0, 1, 400)
plt.figure(figsize=(8, 4))
for n in [0, 10, 50, 200]:
    data = rng.random(n) < true_p
    a_post = alpha0 + data.sum()
    b_post = beta0 + (n - data.sum())
    plt.plot(x, stats.beta.pdf(x, a_post, b_post), label=f'n={n}: Beta({a_post},{b_post})')
plt.axvline(true_p, color='#f87171', ls='--', label='true p=0.30')
plt.xlabel('conversion rate p'); plt.ylabel('posterior density'); plt.legend()
plt.title('Beta posterior concentrates on the truth as data arrives'); plt.tight_layout(); plt.show()

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **assuming Gaussian tails** | underestimates rare events (demo) — check the tails |
| **CLT needs finite variance** | heavy-tailed (Cauchy) means don't converge |
| **discrete vs continuous** | match the distribution to the data's support |
| **small samples** | the CLT is asymptotic; small $m$ isn't Gaussian yet |
| **independence** | the CLT assumes i.i.d.; correlation breaks it |

Demo: a 4-sigma event is far more likely under a heavy-tailed t than a Gaussian.

In [ ]:
# The gotcha that matters for risk: TAILS. Two distributions with the same center can have wildly
# different tail probabilities. A '4-sigma' event is astronomically rare under a Gaussian but
# common under a heavy-tailed Student-t — which is why assuming Gaussianity underestimates rare
# events (financial crashes, latency spikes). We compare the tail mass.
p_gauss = 2 * stats.norm.sf(4)          # P(|Z| > 4) under a Gaussian
p_t = 2 * stats.t.sf(4, df=2)           # under a heavy-tailed t (df=2)
print(f'P(|X| > 4): Gaussian = {p_gauss:.2e},  Student-t(df=2) = {p_t:.4f}')
assert p_t > 10 * p_gauss, 'heavy-tailed distributions make "rare" tail events far more likely'
print('\nAssuming Gaussianity underestimates tail risk -> match the distribution to the data, especially the tails.')

## ✏️ Your turn

**Task A — Overdispersion:** Poisson forces mean = variance. Simulate count data with variance > mean (e.g. a Negative Binomial), fit a Poisson by matching the mean, and show the Poisson under-predicts the tail. Plot both PMFs against the data histogram.

**Task B — Match the support:** Write a helper `suggest_distribution(samples)` that inspects a 1-D array and prints a sensible candidate based on its support and shape (all-integer & nonneg → Poisson/NegBinom; in [0,1] → Beta; positive reals → Gamma/LogNormal; real & symmetric → Gaussian/Student-t). Test it on samples you generate from each family.

In [ ]:
def suggest_distribution(samples):
    s = np.asarray(samples)
    # TODO(you): branch on support (integer? nonnegative? bounded in [0,1]?) and
    # symmetry/skew to print a candidate distribution
    return ...

# quick tests
suggest_distribution(rng.poisson(3, 500))      # expect a count distribution
suggest_distribution(rng.beta(2, 5, 500))      # expect Beta
suggest_distribution(rng.normal(0, 1, 500))    # expect Gaussian/Student-t

<details><summary>Solution — Task B</summary>

```python
def suggest_distribution(samples):
    s = np.asarray(samples)
    is_int = np.allclose(s, np.round(s))
    if is_int and (s >= 0).all():
        verdict = 'Poisson (if mean≈var) or Negative Binomial (if var>mean)'
    elif ((s >= 0) & (s <= 1)).all():
        verdict = 'Beta (a proportion in [0,1])'
    elif (s > 0).all():
        verdict = 'Gamma / Exponential / Log-Normal (positive reals)'
    else:
        skew = stats.skew(s)
        verdict = 'Gaussian' if abs(skew) < 0.5 else 'skewed — consider a transform'
    print(f'support→ {verdict}')
    return verdict
```
</details>

## Key takeaways

- **The families connect:** Binomial → Poisson (rare-event limit, verified), and sample means →
  Gaussian (CLT, verified).
- **Poisson:** mean = variance = $\lambda$ (verified).
- **The CLT** makes averages Gaussian with std $\sigma/\sqrt m$ (verified).
- **Tails differ enormously** (demo) — assuming Gaussianity underestimates rare events.